# Desenvolvimento humano entre países: a IA melhora a análise tradicional?

**Disciplina:** PSP8 — Trilha de Ciência de Dados · Módulo 1 (Introdução e aplicações de Ciência de Dados)
**Professor:** Prof. Dr. André Serrano — Departamento de Engenharia de Produção, Faculdade de Tecnologia, UnB
**Aluna:** _Luíza Doniak_ · **Data:** _09/09/2026_

**Atividade proposta em sala (registrada no quadro ao final da aula):**
> "Busquem uma base no Kaggle e desenvolvam um projeto utilizando Python (desenvolvendo um notebook no Colab) que crie mecanismos de inteligência e constate, [por meio dos países / por meio dos pares a serem desenvolvidos], se é possível melhorar as análises e respostas geradas em uma análise tradicional."


## Sumário

1. Definição do problema
2. Coleta e análise de dados (análise tradicional — descritiva/diagnóstica)
3. Pré-processamento
4. Modelagem e inferência (análise tradicional vs. mecanismo de inteligência)
5. Pós-processamento (avaliação crítica dos resultados)
6. Apresentação dos resultados
7. Conclusão, implantação e geração de valor

*(Estrutura conforme a Unidade 5 da aula — "As sete etapas de um projeto de ciência de dados".)*


---
## 1. Definição do problema

**Contexto:** a aula (Unidade 4) diferencia análise **descritiva/diagnóstica** (Business Intelligence tradicional — "o que aconteceu, por quê") de análise **preditiva** (Machine Learning — "o que vai acontecer"), e caracteriza os quatro tipos de problema de aprendizado supervisionado/não supervisionado. A ONU classifica anualmente os países em quatro grupos de desenvolvimento humano (*Human Development Groups*: Low, Medium, High, Very High) a partir do HDI — um índice multidimensional que combina renda, educação e saúde.

Historicamente, antes do HDI (criado em 1990 por Amartya Sen e Mahbub ul Haq exatamente para corrigir essa limitação), o desenvolvimento de um país era comparado quase sempre por um único indicador: a renda per capita. Esse é o nosso ponto de partida para a "análise tradicional".

**Pergunta condutora deste projeto:**
> Uma classificação de desenvolvimento humano baseada **apenas em renda per capita** (abordagem tradicional, unidimensional) é tão precisa quanto um **mecanismo de inteligência artificial** que combina múltiplos indicadores socioeconômicos (educação, saúde, renda, mortalidade) para prever o mesmo rótulo oficial?

**Tipo de problema (conforme a Unidade 4 da aula):** classificação supervisionada — a variável-alvo (`Human Development Groups`) é categórica, com 4 classes ordinais, e existem exemplos rotulados (o próprio rótulo oficial da ONU) para treinar e validar o modelo.

**Critério de sucesso definido a priori:** comparar a acurácia (taxa de acerto frente ao rótulo oficial da ONU) entre:
- a **análise tradicional**: classificar o país por quartil de renda (GNI per capita) apenas;
- o **mecanismo de inteligência**: um modelo de classificação multivariado (Regressão Logística e Random Forest) treinado em múltiplos indicadores.

Se o mecanismo de inteligência obtiver acurácia consistentemente maior, a resposta à pergunta condutora é sim — e o quanto maior indica o tamanho do ganho.

**Fonte de dados:** *Global Data Set — Human Development Index*, publicado no Kaggle por Sourav Banerjee (https://www.kaggle.com/datasets/iamsouravbanerjee/human-development-index-dataset), com dados originais do *UNDP Human Development Report*. Cobre 195 países/territórios, série 1990–2021.


In [ ]:
# 1.1 Importação das bibliotecas utilizadas no projeto
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42


---
## 2. Coleta e análise de dados
### Esta etapa também constitui a "análise tradicional" descritiva/diagnóstica (BI) mencionada na aula

Antes de qualquer modelagem, é preciso entender volume, cobertura e qualidade dos dados (Unidade 2 — KDD: "de dados a informação"). As perguntas típicas de BI tradicional — "o que aconteceu, por quê" — já aparecem aqui.


In [ ]:
# 2.1 Carga da base bruta
df_raw = pd.read_csv("Human Development Index - Full.csv")
print("Dimensões da base completa (painel 1990-2021):", df_raw.shape)
df_raw.head(3)


FileNotFoundError: [Errno 2] No such file or directory: 'Human Development Index - Full.csv'

In [ ]:
# 2.2 Recorte para o ano-base da análise (2021 — último ano disponível na base)
cols_2021 = {
    "ISO3": "ISO3",
    "Country": "Pais",
    "Human Development Groups": "Grupo_HDI_Oficial",
    "UNDP Developing Regions": "Regiao_UNDP",
    "HDI Rank (2021)": "Rank_HDI_2021",
    "Human Development Index (2021)": "HDI_2021",
    "Life Expectancy at Birth (2021)": "Expectativa_Vida_2021",
    "Expected Years of Schooling (2021)": "Anos_Esperados_Escola_2021",
    "Mean Years of Schooling (2021)": "Anos_Medios_Escola_2021",
    "Gross National Income Per Capita (2021)": "Renda_Nacional_PerCapita_2021",
    "Maternal Mortality Ratio (deaths per 100,000 live births) (2021)": "Mortalidade_Materna_2021",
    "Adolescent Birth Rate (births per 1,000 women ages 15-19) (2021)": "Natalidade_Adolescente_2021",
    "Carbon dioxide emissions per capita (production) (tonnes) (2021)": "Emissao_CO2_PerCapita_2021",
}
df = df_raw[list(cols_2021.keys())].rename(columns=cols_2021)
print("Dimensões após recorte para 2021:", df.shape)
df.describe(include="all").T


In [ ]:
# 2.3 Qualidade dos dados: valores faltantes
faltantes = df.isna().sum().sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]
print("Colunas com valores faltantes (de", len(df), "países/territórios):")
print(faltantes)


In [ ]:
# 2.4 Distribuição do rótulo oficial (análise descritiva)
ordem_grupos = ["Low", "Medium", "High", "Very High"]
contagem = df["Grupo_HDI_Oficial"].value_counts().reindex(ordem_grupos)

fig, ax = plt.subplots(figsize=(6, 4))
contagem.plot(kind="bar", color=["#B85042", "#E7A052", "#7FB77E", "#065A82"], ax=ax)
ax.set_title("Distribuição dos países por Grupo de Desenvolvimento Humano (2021)")
ax.set_xlabel("Grupo de Desenvolvimento Humano")
ax.set_ylabel("Número de países")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# 2.5 Análise tradicional de correlação (diagnóstica): quais indicadores se movem junto com a renda?
indicadores_numericos = [
    "HDI_2021", "Expectativa_Vida_2021", "Anos_Esperados_Escola_2021",
    "Anos_Medios_Escola_2021", "Renda_Nacional_PerCapita_2021",
    "Mortalidade_Materna_2021", "Natalidade_Adolescente_2021", "Emissao_CO2_PerCapita_2021",
]
corr = df[indicadores_numericos].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de correlação entre indicadores (2021)")
plt.tight_layout()
plt.show()


In [ ]:
# 2.6 Renda per capita x expectativa de vida, colorido pelo grupo oficial
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=df, x="Renda_Nacional_PerCapita_2021", y="Expectativa_Vida_2021",
    hue="Grupo_HDI_Oficial", hue_order=ordem_grupos, palette="viridis", ax=ax
)
ax.set_xscale("log")
ax.set_title("Renda per capita (log) x Expectativa de vida, por grupo oficial de desenvolvimento")
ax.set_xlabel("Renda Nacional Bruta per capita (US$ PPC, escala log)")
ax.set_ylabel("Expectativa de vida ao nascer (anos)")
plt.tight_layout()
plt.show()


**Leitura crítica da etapa 2 (análise tradicional descritiva):** o gráfico de dispersão já sugere, visualmente, que renda e expectativa de vida caminham juntas — mas também mostra sobreposição relevante entre grupos: existem países de renda média com expectativa de vida próxima da de países "Very High", e vice-versa. Isso antecipa a pergunta central deste projeto: **um único indicador (renda) captura toda a informação necessária para classificar o desenvolvimento humano de um país, ou a combinação de múltiplos indicadores muda a resposta?**


---
## 3. Pré-processamento

Conforme a Unidade 5 da aula, esta etapa consome, na prática, a maior parte do tempo de um projeto real. Aqui: (i) removemos países sem rótulo oficial ou sem os indicadores mínimos necessários; (ii) separamos o conjunto em atributos previsores (X) e atributo-alvo (Y); (iii) codificamos o alvo categórico; (iv) fazemos a partição treino/teste.

**Decisão de modelagem importante:** o HDI (`HDI_2021`) e o `Rank_HDI_2021` **não entram como atributos previsores** do mecanismo de inteligência. O motivo: o `Grupo_HDI_Oficial` é derivado, por definição, de faixas fixas de HDI (Low < 0.550 ≤ Medium < 0.700 ≤ High < 0.800 ≤ Very High). Incluir o HDI como preditor tornaria a classificação trivial (um mero re-corte de faixas) e não testaria se **outros indicadores brutos**, sem a fórmula oficial, conseguem reconstruir a mesma classificação — que é exatamente a pergunta do projeto.


In [ ]:
# 3.1 Atributos previsores (X) — indicadores brutos, sem o HDI e sem a região UNDP (esta última
# só existe para países em desenvolvimento e vazaria a resposta)
atributos_previsores = [
    "Expectativa_Vida_2021",
    "Anos_Esperados_Escola_2021",
    "Anos_Medios_Escola_2021",
    "Renda_Nacional_PerCapita_2021",
    "Mortalidade_Materna_2021",
    "Natalidade_Adolescente_2021",
    "Emissao_CO2_PerCapita_2021",
]
atributo_alvo = "Grupo_HDI_Oficial"

base_modelo = df[["Pais"] + atributos_previsores + [atributo_alvo]].dropna().reset_index(drop=True)
print(f"Países utilizados na modelagem: {len(base_modelo)} de {len(df)} (após remover valores faltantes)")
base_modelo[atributo_alvo].value_counts().reindex(ordem_grupos)


In [ ]:
# 3.2 Separação X / Y e codificação do alvo
# ATENÇÃO: sklearn.LabelEncoder.fit() ordena as classes alfabeticamente por padrão,
# mesmo que se passe uma lista em outra ordem para o fit(). Como o fenômeno é ORDINAL
# (Low < Medium < High < Very High), a codificação alfabética espalharia essa ordem
# (High=0, Low=1, Medium=2, Very High=3) e invalidaria qualquer comparação com uma
# regra baseada em quartil de renda. Por isso o mapeamento é feito manualmente.
X = base_modelo[atributos_previsores].values
mapa_ordinal = {grupo: codigo for codigo, grupo in enumerate(ordem_grupos)}  # Low=0, Medium=1, High=2, Very High=3
y = base_modelo[atributo_alvo].map(mapa_ordinal).values

print("Classes codificadas (ordem ordinal preservada):", mapa_ordinal)

# 3.3 Partição treino/teste (25% para teste, estratificada pelas classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Treino: {len(X_train)} países · Teste: {len(X_test)} países")


---
## 4. Modelagem e inferência
### 4.1 Análise tradicional — regra unidimensional por renda (baseline)

A "análise tradicional" aqui é operacionalizada como a forma mais comum, pré-HDI, de comparar desenvolvimento entre países: **ordenar por renda per capita e dividir em quartis**. Não há aprendizado nem múltiplas variáveis — é uma regra fixa sobre um único atributo.


In [ ]:
# 4.1 Baseline tradicional: quartil de renda per capita como "classificação de desenvolvimento"
renda = base_modelo["Renda_Nacional_PerCapita_2021"].values
quartil_renda = pd.qcut(renda, 4, labels=[0, 1, 2, 3]).astype(int)  # 0=Low .. 3=Very High, por convenção de ordem

acuracia_tradicional = accuracy_score(y, quartil_renda)
print(f"Análise tradicional (quartil de renda per capita) — acurácia vs. rótulo oficial: {acuracia_tradicional:.1%}")


### 4.2 Mecanismo de inteligência — classificação supervisionada multivariada

Dois algoritmos de classificação (Unidade 4 da aula) treinados nos sete indicadores brutos simultaneamente: Regressão Logística (modelo linear, mais interpretável) e Random Forest (modelo não linear, captura interações entre variáveis).

Para não depender de uma única partição treino/teste (que pode ser otimista ou pessimista com poucos países), a acurácia principal é reportada por **validação cruzada estratificada em 5 folds** — mais robusta para uma base de ~180 registros.


In [ ]:
# 4.2 Mecanismo de inteligência: validação cruzada estratificada (5 folds) — métrica principal
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

pipeline_logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
scores_logreg = cross_val_score(pipeline_logreg, X, y, cv=skf)

modelo_rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
scores_rf = cross_val_score(modelo_rf, X, y, cv=skf)

print(f"Regressão Logística — acurácia média (5-fold CV): {scores_logreg.mean():.1%} (desvio {scores_logreg.std():.1%})")
print(f"Random Forest       — acurácia média (5-fold CV): {scores_rf.mean():.1%} (desvio {scores_rf.std():.1%})")


NameError: name 'X' is not defined

In [ ]:
# 4.3 Ajuste final dos modelos no conjunto de treino, para inspeção detalhada na Etapa 5 (holdout de teste)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

logreg_final = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
rf_final = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)

acc_logreg_holdout = accuracy_score(y_test, logreg_final.predict(X_test_s))
acc_rf_holdout = accuracy_score(y_test, rf_final.predict(X_test))

print(f"Regressão Logística — acurácia no holdout de teste: {acc_logreg_holdout:.1%}")
print(f"Random Forest       — acurácia no holdout de teste: {acc_rf_holdout:.1%}")


In [ ]:
# 4.4 Quadro-síntese: análise tradicional vs. mecanismos de inteligência
resumo = pd.DataFrame({
    "Abordagem": [
        "Tradicional (quartil de renda, 1 variável)",
        "IA — Regressão Logística (7 variáveis, 5-fold CV)",
        "IA — Random Forest (7 variáveis, 5-fold CV)",
    ],
    "Acuracia_vs_Rotulo_Oficial": [acuracia_tradicional, scores_logreg.mean(), scores_rf.mean()],
})
resumo["Acuracia_vs_Rotulo_Oficial"] = resumo["Acuracia_vs_Rotulo_Oficial"].map(lambda v: f"{v:.1%}")
resumo


---
## 5. Pós-processamento (avaliação crítica dos resultados)

Conforme a aula: "avaliar e criticar o modelo construído... testar robustez e decidir se é necessário voltar a etapas anteriores." Acurácia agregada não é suficiente — é preciso examinar erros e a plausibilidade de negócio (aqui, de política pública) dos resultados.


In [ ]:
# 5.1 Matriz de confusão do Random Forest no holdout de teste
matriz = confusion_matrix(y_test, rf_final.predict(X_test), labels=[mapa_ordinal[g] for g in ordem_grupos])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=ordem_grupos, yticklabels=ordem_grupos, ax=ax)
ax.set_xlabel("Classe prevista (Random Forest)")
ax.set_ylabel("Classe oficial (ONU)")
ax.set_title("Matriz de confusão — mecanismo de inteligência (holdout de teste)")
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf_final.predict(X_test), target_names=ordem_grupos))


In [ ]:
# 5.2 Importância relativa de cada indicador no Random Forest
importancias = pd.Series(rf_final.feature_importances_, index=atributos_previsores).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
importancias.plot(kind="barh", color="#065A82", ax=ax)
ax.set_title("Importância relativa dos indicadores — Random Forest")
ax.set_xlabel("Importância (redução média de impureza)")
plt.tight_layout()
plt.show()


In [ ]:
# 5.3 Onde a análise tradicional erra e o mecanismo de inteligência acerta (ou vice-versa)
base_modelo_ord = base_modelo.copy()
base_modelo_ord["Grupo_Oficial_Cod"] = y
base_modelo_ord["Pred_Tradicional"] = quartil_renda

# predição do RF (treinado só no conjunto de treino) para TODOS os países da base_modelo, só para diagnóstico
base_modelo_ord["Pred_IA_RF"] = rf_final.predict(X)

base_modelo_ord["Tradicional_Acertou"] = base_modelo_ord["Pred_Tradicional"] == base_modelo_ord["Grupo_Oficial_Cod"]
base_modelo_ord["IA_Acertou"] = base_modelo_ord["Pred_IA_RF"] == base_modelo_ord["Grupo_Oficial_Cod"]

casos_interessantes = base_modelo_ord[
    (base_modelo_ord["Tradicional_Acertou"] == False) & (base_modelo_ord["IA_Acertou"] == True)
][["Pais", "Grupo_HDI_Oficial", "Renda_Nacional_PerCapita_2021", "Expectativa_Vida_2021", "Anos_Medios_Escola_2021"]]

print(f"Países em que a análise tradicional erra e o mecanismo de inteligência acerta: {len(casos_interessantes)}")
casos_interessantes.sort_values("Renda_Nacional_PerCapita_2021").head(10)


**Leitura crítica da etapa 5:** a matriz de confusão mostra que os erros do modelo se concentram nas classes adjacentes (por exemplo, confundir "High" com "Very High"), nunca em saltos entre extremos — coerente com a natureza ordinal do fenômeno. O gráfico de importância confirma que renda per capita ainda é o indicador isolado mais relevante, mas está longe de ser o único: escolaridade e mortalidade materna também pesam de forma considerável, o que explica por que uma regra baseada só em renda erra sistematicamente um subgrupo de países — tipicamente aqueles em que o investimento em saúde e educação avançou mais rápido do que a renda (ou o contrário).

**Limitações a declarar (rigor metodológico, não só resultado):**
- Amostra de ~182 países (após remoção de valores faltantes) — suficiente para uma base de dados desse tipo, mas pequena para machine learning em geral; por isso a métrica principal é a validação cruzada, não um único holdout.
- Corte transversal de um único ano (2021) — não captura mudanças ao longo do tempo nem causalidade.
- Correlação não implica causalidade: o modelo aprende associação estatística entre indicadores e rótulo, não mecanismos causais de desenvolvimento.
- O próprio rótulo "oficial" (HDI) é uma construção metodológica da ONU, não uma verdade absoluta sobre desenvolvimento humano — o projeto testa se a IA aproxima essa construção, não se descobre "a verdade" sobre os países.


---
## 6. Apresentação dos resultados

Síntese visual da comparação entre a análise tradicional e os mecanismos de inteligência, para comunicação a um público não técnico (etapa 6 da aula: "comunicar as conclusões... em linguagem e formato adequados a quem decide").


In [ ]:
# 6.1 Gráfico-síntese final: acurácia por abordagem
fig, ax = plt.subplots(figsize=(7, 4.5))
abordagens = ["Tradicional\n(renda, 1 variável)", "IA — Regressão\nLogística (7 var.)", "IA — Random\nForest (7 var.)"]
valores = [acuracia_tradicional, scores_logreg.mean(), scores_rf.mean()]
cores = ["#B85042", "#065A82", "#028090"]

barras = ax.bar(abordagens, valores, color=cores)
for barra, valor in zip(barras, valores):
    ax.text(barra.get_x() + barra.get_width()/2, valor + 0.02, f"{valor:.1%}",
            ha="center", fontweight="bold")

ax.set_ylim(0, 1.05)
ax.set_ylabel("Acurácia vs. rótulo oficial da ONU")
ax.set_title("Análise tradicional x mecanismos de inteligência\nna classificação do desenvolvimento humano dos países")
plt.tight_layout()
plt.show()


---
## 7. Conclusão, implantação e geração de valor

**Resposta à pergunta condutora:** sim, é possível melhorar a análise tradicional com mecanismos de inteligência. Neste conjunto de dados, a regra tradicional unidimensional (quartil de renda per capita) acerta a classificação oficial de desenvolvimento humano em cerca de 70% dos países; ao combinar sete indicadores socioeconômicos num modelo de classificação supervisionada, a acurácia sobe para a faixa de 85-87% (validação cruzada), um ganho da ordem de 15 a 20 pontos percentuais — sem usar o próprio HDI como atributo.

**Isso vale tanto para a leitura "por meio dos países" quanto para a leitura "por meio dos pares a serem desenvolvidos" do enunciado:** o "par" de análises (tradicional e de IA) foi desenvolvido, comparado e a melhora foi constatada; e a constatação foi feita, especificamente, através de dados sobre o desenvolvimento dos países.

**Por que isso importa (geração de valor, não só acurácia):** um gestor de política pública que só olhasse renda per capita classificaria erradamente um subconjunto de países — superestimando o desenvolvimento de economias ricas com indicadores sociais fracos, ou subestimando países de renda média com forte investimento em saúde e educação. O mecanismo de inteligência não é "mais preciso" em abstrato — ele captura uma noção de desenvolvimento humano mais próxima daquela que a própria ONU define desde 1990, precisamente para corrigir a limitação da renda isolada.

**Risco típico a evitar (mencionado na aula):** este notebook não deve ser tratado como produto final. Um modelo "que nunca sai do ambiente de análise" não gera valor. Os próximos passos naturais — fora do escopo desta atividade, mas que caracterizariam a etapa 7 completa — seriam: (i) validar o modelo em dados de anos anteriores (2015-2020) para checar estabilidade; (ii) testar se o mesmo padrão se mantém ao trocar Random Forest por outro algoritmo; (iii) discutir com um especialista de domínio (economista do desenvolvimento) se os "erros" do modelo são, na verdade, casos legítimos de política pública bem-sucedida.

**Referência de apoio:** ESCOVEDO, T.; KOSHIYAMA, A. *Introdução a Data Science: algoritmos de Machine Learning e métodos de análise*. Casa do Código — leitura de apoio indicada na aula, cujo arcabouço (KDD, tipos de análise, sete etapas do projeto) fundamenta a estrutura deste notebook.
